# 🧬 Classification with k-NN and Naïve Bayes: From Scratch to Clinical-Style Evaluation

**Subtitle:** Machine Learning for Scientists, Topic 3 — an instance-based vs. a generative classifier on real tumour-measurement data

| | |
|---|---|
| **Hardware Target** | CPU only (no GPU needed), Google Colab free tier |
| **Core Frameworks** | NumPy, scikit-learn, Matplotlib |
| **Dataset** | Wisconsin Diagnostic Breast Cancer (569 samples, 30 features), bundled with scikit-learn, so no download is needed |
| **Prerequisites** | Basic Python/NumPy, probability (Bayes' rule), train/test splits |
| **Estimated Runtime** | ~1–2 minutes |

> ⚠️ **Disclaimer:** For research and educational purposes only, not to be used for clinical diagnosis.

---

## 🎯 What You Will Learn

1. How **k-NN** (instance-based, non-parametric) and **Gaussian Naïve Bayes** (generative, parametric) make decisions, and how their decision boundaries differ.
2. How to **implement both from scratch in NumPy** and verify them against scikit-learn.
3. Why **feature scaling** is essential for k-NN but irrelevant for Naïve Bayes.
4. How to **choose $k$ properly** with cross-validation (and the one-standard-error rule), without touching the test set.
5. How both models behave as you add **irrelevant features** (the curse of dimensionality) and **redundant, correlated features** (the independence assumption).
6. How to evaluate beyond accuracy: **confusion matrices, ROC/AUC, calibration, threshold choice** and **learning curves**.

## 📋 Table of Contents

1. Environment Setup
2. Theory: k-NN and Naïve Bayes
3. Intuition in 2D: Decision Boundaries
4. Real Data: Wisconsin Breast Cancer
5. Implementing Both Classifiers From Scratch
6. Feature Scaling
7. Choosing $k$ with Cross-Validation
8. Irrelevant Features: The Curse of Dimensionality
9. Correlated Features: When "Naïve" Hurts
10. Beyond Accuracy: Confusion Matrix, ROC/AUC, Threshold
11. Probability Calibration
12. Learning Curves
13. Summary
14. Questions & Bonus Exercises

## 1. Environment Setup

In [ ]:
# ── Hardware Check ──
import os, sys, platform
print(f"Python {sys.version.split()[0]} | {platform.system()} | CPU cores: {os.cpu_count()}")
print("This tutorial needs no GPU. CPU mode is fine ✅")

In [ ]:
# ── Install Dependencies ──
# Colab already ships these; the line is here so the notebook is self-contained.
!pip install -q numpy scikit-learn matplotlib

In [ ]:
# ── Global Imports & Configuration ──
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.datasets import load_breast_cancer, make_classification, make_moons, make_circles
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_val_score, cross_val_predict, learning_curve)
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, roc_auc_score,
                             brier_score_loss, log_loss)
from sklearn.calibration import calibration_curve

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 12, "axes.labelsize": 11})

C_KNN, C_NB = "#d95f02", "#1b9e77"   # one colour per model, reused in every plot
print("✅ Imports OK")

## 2. Theory: k-NN and Naïve Bayes

### 2.1 k-Nearest Neighbours (instance-based, non-parametric)

k-NN has **no training step**: it memorises the training set. To classify a new point $\mathbf{x}$ it finds the $k$ closest training points under the Euclidean distance

$$d(\mathbf{x}, \mathbf{x}_i) = \sqrt{\sum_{j=1}^{p} (x_j - x_{ij})^2}$$

and takes a vote among them:

$$\hat{P}(Y=c \mid \mathbf{x}) = \frac{1}{k} \sum_{i \in \mathcal{N}_k(\mathbf{x})} \mathbb{1}[y_i = c], \qquad \hat{y} = \arg\max_c \hat{P}(Y=c \mid \mathbf{x})$$

* **Small $k$** → very flexible boundary, low bias, **high variance** (overfits noise).
* **Large $k$** → smooth boundary, high bias, low variance (can miss structure).
* **Costs:** all work happens at prediction time ($O(n p)$ per query for brute force), and *distances depend on feature scales*.

### 2.2 Gaussian Naïve Bayes (generative, parametric)

Naïve Bayes models how each class *generates* the data and inverts it with Bayes' rule:

$$P(Y=c \mid \mathbf{x}) \;\propto\; P(Y=c) \prod_{j=1}^{p} P(x_j \mid Y=c)$$

The **naïve assumption** is that features are *conditionally independent given the class*. This replaces one $p$-dimensional density per class with $p$ one-dimensional ones, which avoids parameter explosion. For continuous features, each factor is a Gaussian:

$$P(x_j \mid Y=c) = \frac{1}{\sqrt{2\pi\sigma_{c,j}^2}} \exp\!\left(-\frac{(x_j-\mu_{c,j})^2}{2\sigma_{c,j}^2}\right)$$

"Training" is just computing $\pi_c$, $\mu_{c,j}$, $\sigma^2_{c,j}$ — closed form, one pass over the data. In practice we work in **log space** to avoid underflow from multiplying many small numbers:

$$\log P(Y=c \mid \mathbf{x}) = \log \pi_c - \tfrac12 \sum_{j}\Big[\log(2\pi\sigma_{c,j}^2) + \tfrac{(x_j-\mu_{c,j})^2}{\sigma_{c,j}^2}\Big] - \log Z(\mathbf{x})$$

### 2.3 Side by side

| | k-NN | Gaussian Naïve Bayes |
|---|---|---|
| **Type** | Instance-based, non-parametric | Generative, parametric |
| **Training cost** | None (stores data) | One pass (means, variances) |
| **Prediction cost** | High: scans all training points | Low: $O(Cp)$ |
| **Key assumption** | Nearby points share labels; distance is meaningful | Features independent given the class; Gaussian shape |
| **Needs scaling?** | **Yes** | No |
| **Main hyper-parameter** | $k$ (and the distance metric) | Variance smoothing (rarely tuned) |

## 3. Intuition in 2D: Decision Boundaries

Before using real data, we look at three small 2D problems where we can *see* what each model assumes. This is the original demo from this notebook, extended with two datasets where the assumptions of one model break.

In [ ]:
# ── 2D Toy Datasets ──
def make_toy_datasets(seed=SEED):
    '''Return {name: (X, y)}: one dataset suited to Gaussian NB, two that violate its assumptions.'''
    gauss = make_classification(n_samples=300, n_features=2, n_informative=2, n_redundant=0,
                                n_clusters_per_class=1, class_sep=1.2, random_state=seed)
    moons = make_moons(n_samples=300, noise=0.25, random_state=seed)
    circles = make_circles(n_samples=300, noise=0.12, factor=0.45, random_state=seed)
    return {"Gaussian blobs": gauss, "Two moons": moons, "Concentric circles": circles}

def plot_decision_boundary(clf, X, y, ax, title):
    '''Shade the predicted class over a grid and overlay the data points.'''
    x_min, x_max = X[:, 0].min() - 0.7, X[:, 0].max() + 0.7
    y_min, y_max = X[:, 1].min() - 0.7, X[:, 1].max() + 0.7
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(["#FFAAAA", "#AAAAFF"]))
    for c, col in zip([0, 1], ["red", "blue"]):
        ax.scatter(X[y == c, 0], X[y == c, 1], c=col, edgecolors="k", s=20, alpha=0.7)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

toys = make_toy_datasets()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, (name, (X2, y2)) in enumerate(toys.items()):
    X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.3, stratify=y2, random_state=SEED)
    for row, (label, clf) in enumerate([("k-NN (k=7)", KNeighborsClassifier(7)), ("Gaussian NB", GaussianNB())]):
        clf.fit(X2_tr, y2_tr)
        plot_decision_boundary(clf, X2_tr, y2_tr, axes[row, col],
                               f"{name}\n{label}: test acc = {clf.score(X2_te, y2_te):.2f}")
plt.suptitle("Decision boundaries: k-NN (top) vs. Gaussian Naïve Bayes (bottom)", fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

**What to notice**

* On Gaussian blobs both models do well, because Naïve Bayes' assumptions hold.
* On the **moons**, the classes are not Gaussian blobs. Gaussian NB can only draw smooth (at most quadratic) boundaries, so it cuts the interleaved crescents with a nearly straight line, and k-NN, which follows the shape of the data, does better.
* The **concentric circles** are a useful surprise: Gaussian NB does fine here. Both classes have the same centre but *different variances*, and a class-specific variance lets NB draw an elliptical (quadratic) boundary. NB's failure depends on the *shape* of the classes, not on whether they look "non-linear".
* This is the **bias–variance trade-off** in action: NB has high bias but is stable; k-NN has low bias but its quality depends on $k$ (next cell).

In [ ]:
# ── Effect of k on the Boundary (two moons) ──
X2, y2 = toys["Two moons"]
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.3, stratify=y2, random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, k in zip(axes, [1, 15, 100]):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X2_tr, y2_tr)
    plot_decision_boundary(knn, X2_tr, y2_tr, ax,
        f"k = {k}\ntrain acc = {knn.score(X2_tr, y2_tr):.2f} | test acc = {knn.score(X2_te, y2_te):.2f}")
plt.suptitle("k=1 memorises noise (train acc = 1.00); very large k over-smooths", y=1.02)
plt.tight_layout()
plt.show()

## 4. Real Data: Wisconsin Breast Cancer

Each of the 569 samples describes a **fine-needle-aspirate image of a breast mass**. Ten nuclear properties (radius, texture, perimeter, area, smoothness, compactness, concavity, concave points, symmetry, fractal dimension) are computed per image, and for each we get the **mean, standard error and "worst" (largest)** value, giving 30 features. The task: **malignant (1) vs. benign (0)**.

We relabel the target so that *malignant = 1*, the "positive" (disease) class. This matters for recall, ROC curves and false-negative counts later.

**Evaluation protocol (fixed from the start to avoid leakage):**

1. Split off a **25% test set** now and do not touch it until the final evaluation.
2. Do *all* model selection (choice of $k$, feature-scaling decision, etc.) with **cross-validation on the training portion only**.
3. Always put the scaler *inside* a `Pipeline`, so it is re-fit on each training fold.

In [ ]:
# ── Load Data & Hold Out a Test Set ──
data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)          # 1 = malignant, 0 = benign
feature_names = list(data.feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]}")
print(f"Malignant fraction  — all: {y.mean():.3f} | train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")
print(f"Train: {len(y_train)} | Test: {len(y_test)}")

# CV scheme reused throughout: 5 folds x 3 repeats = 15 estimates per setting (less noisy than 1 x 5).
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)

In [ ]:
# ── Explore the Features ──
# 1) Scale differences: this is why k-NN will need standardisation.
ranges = X_train.max(axis=0) - X_train.min(axis=0)
order = np.argsort(ranges)
print("Smallest feature range:", feature_names[order[0]], f"= {ranges[order[0]]:.5f}")
print("Largest  feature range:", feature_names[order[-1]], f"= {ranges[order[-1]]:.1f}")

# 2) Class-conditional distributions of four features + correlation structure.
fig, axes = plt.subplots(1, 5, figsize=(20, 3.8))
for ax, name in zip(axes[:4], ["mean radius", "mean texture", "mean concave points", "worst area"]):
    j = feature_names.index(name)
    ax.hist(X_train[y_train == 0, j], bins=20, alpha=0.6, color="steelblue", label="benign")
    ax.hist(X_train[y_train == 1, j], bins=20, alpha=0.6, color="crimson", label="malignant")
    ax.set_title(name, fontsize=10)
axes[0].legend()
corr = np.corrcoef(X_train, rowvar=False)
im = axes[4].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[4].set_title("Feature correlation (30x30)", fontsize=10)
plt.colorbar(im, ax=axes[4], fraction=0.046)
plt.tight_layout(); plt.show()

n_high = (np.abs(corr[np.triu_indices(30, 1)]) > 0.9).sum()
print(f"Feature pairs with |r| > 0.9: {n_high}  → the 'conditional independence' assumption is clearly violated here.")

## 5. Implementing Both Classifiers From Scratch

Writing the algorithms yourself is the best way to understand them. We implement both with plain NumPy and then **verify they match scikit-learn**.

### 5.1 k-NN

The only subtle part is computing all pairwise distances efficiently. Using $\|\mathbf{a}-\mathbf{b}\|^2 = \|\mathbf{a}\|^2 + \|\mathbf{b}\|^2 - 2\,\mathbf{a}\cdot\mathbf{b}$ turns it into one matrix multiplication instead of a Python double loop.

In [ ]:
# ── k-NN From Scratch ──
class KNNScratch:
    '''Brute-force k-nearest-neighbour classifier (Euclidean distance, uniform vote).'''

    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_ = np.asarray(X, dtype=float)
        self.classes_, self.y_idx_ = np.unique(y, return_inverse=True)   # labels -> 0..C-1
        return self

    def _sq_dists(self, Xq):
        '''Squared Euclidean distance between every query and every training point.'''
        d2 = (Xq**2).sum(1)[:, None] + (self.X_**2).sum(1)[None, :] - 2 * Xq @ self.X_.T
        return np.maximum(d2, 0)          # clip tiny negatives caused by floating point

    def predict_proba(self, Xq):
        Xq = np.asarray(Xq, dtype=float)
        nn = np.argpartition(self._sq_dists(Xq), self.k - 1, axis=1)[:, :self.k]   # indices of k nearest
        votes = self.y_idx_[nn]                                                    # (n_query, k) labels
        return np.stack([(votes == c).mean(1) for c in range(len(self.classes_))], axis=1)

    def predict(self, Xq):
        return self.classes_[self.predict_proba(Xq).argmax(1)]

### 5.2 Gaussian Naïve Bayes

Training computes a prior, a mean and a variance per class and feature. Prediction sums log-likelihoods and normalises with the **log-sum-exp trick** (subtract the max before exponentiating) so the posteriors stay numerically stable. We add a tiny variance floor (`var_smoothing`), like scikit-learn does, so a constant feature never causes a division by zero.

In [ ]:
# ── Gaussian Naïve Bayes From Scratch ──
class GaussianNBScratch:
    '''Gaussian Naive Bayes with log-space inference and a variance floor.'''

    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self.classes_ = np.unique(y)
        self.prior_ = np.array([(y == c).mean() for c in self.classes_])
        self.mu_ = np.array([X[y == c].mean(0) for c in self.classes_])           # (C, p)
        self.var_ = np.array([X[y == c].var(0) for c in self.classes_])           # (C, p), MLE (ddof=0)
        self.var_ += self.var_smoothing * X.var(0).max()                          # floor, same rule as sklearn
        return self

    def _joint_log_likelihood(self, Xq):
        '''log P(c) + sum_j log N(x_j | mu_cj, var_cj) for every sample and class.'''
        Xq = np.asarray(Xq, dtype=float)
        jll = []
        for c in range(len(self.classes_)):
            log_gauss = -0.5 * (np.log(2 * np.pi * self.var_[c]) + (Xq - self.mu_[c])**2 / self.var_[c])
            jll.append(np.log(self.prior_[c]) + log_gauss.sum(1))
        return np.stack(jll, axis=1)                                              # (n, C)

    def predict_proba(self, Xq):
        jll = self._joint_log_likelihood(Xq)
        jll -= jll.max(1, keepdims=True)                                          # log-sum-exp trick
        p = np.exp(jll)
        return p / p.sum(1, keepdims=True)

    def predict(self, Xq):
        return self.classes_[self._joint_log_likelihood(Xq).argmax(1)]

### 5.3 Verification against scikit-learn

A from-scratch implementation is only trustworthy if it agrees with a reference. We compare **predictions** and **probabilities** on the (scaled) test set.

In [ ]:
# ── Verify Scratch == scikit-learn ──
scaler = StandardScaler().fit(X_train)                    # fit on TRAIN only
Xtr_s, Xte_s = scaler.transform(X_train), scaler.transform(X_test)

# k-NN
mine_knn = KNNScratch(k=7).fit(Xtr_s, y_train)
ref_knn  = KNeighborsClassifier(n_neighbors=7).fit(Xtr_s, y_train)
print("k-NN  predictions identical :", np.array_equal(mine_knn.predict(Xte_s), ref_knn.predict(Xte_s)))
print("k-NN  probabilities close   :", np.allclose(mine_knn.predict_proba(Xte_s), ref_knn.predict_proba(Xte_s)))

# Gaussian NB
mine_nb = GaussianNBScratch().fit(Xtr_s, y_train)
ref_nb  = GaussianNB().fit(Xtr_s, y_train)
print("NB    predictions identical :", np.array_equal(mine_nb.predict(Xte_s), ref_nb.predict(Xte_s)))
print("NB    probabilities close   :", np.allclose(mine_nb.predict_proba(Xte_s), ref_nb.predict_proba(Xte_s)))
print(f"\nTest accuracy — k-NN (scratch): {accuracy_score(y_test, mine_knn.predict(Xte_s)):.4f} | "
      f"NB (scratch): {accuracy_score(y_test, mine_nb.predict(Xte_s)):.4f}")
assert np.array_equal(mine_knn.predict(Xte_s), ref_knn.predict(Xte_s))
assert np.allclose(mine_nb.predict_proba(Xte_s), ref_nb.predict_proba(Xte_s))

> *These accuracies are on the test set and are shown only to confirm the implementations agree. Do not use them to tune anything; model selection happens with cross-validation on the training set in the next sections.*

## 6. Feature Scaling

k-NN's distance is dominated by whichever features have the largest numeric range, here "area" features in the hundreds-to-thousands vs. "smoothness" around 0.1. Naïve Bayes models each feature **separately** with its own mean and variance, so rescaling a feature changes (almost) nothing. Let us measure both claims with cross-validation.

In [ ]:
# ── Scaling: k-NN vs. NB, with and without StandardScaler ──
def cv_acc(model, Xm=X_train, ym=y_train):
    s = cross_val_score(model, Xm, ym, cv=cv, scoring="accuracy")
    return s.mean(), s.std()

rows = {
    "k-NN (k=7), raw features":    KNeighborsClassifier(7),
    "k-NN (k=7), standardised":    make_pipeline(StandardScaler(), KNeighborsClassifier(7)),
    "Gaussian NB, raw features":   GaussianNB(),
    "Gaussian NB, standardised":   make_pipeline(StandardScaler(), GaussianNB()),
}
print(f"{'Model':<32}{'CV accuracy':>14}")
print("-" * 46)
scale_res = {}
for name, model in rows.items():
    m, s = cv_acc(model); scale_res[name] = m
    print(f"{name:<32}{m:>8.4f} ± {s:.3f}")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(list(scale_res)[::-1], list(scale_res.values())[::-1],
        color=[C_NB, C_NB, C_KNN, C_KNN], alpha=0.85)
ax.set_xlim(0.8, 1.0); ax.set_xlabel("5x3 repeated CV accuracy"); ax.set_title("Scaling matters for k-NN, not for Naïve Bayes")
plt.tight_layout(); plt.show()

**Takeaway:** k-NN accuracy changes noticeably with standardisation; Gaussian NB gives essentially the *same* number either way (a per-feature affine transform just re-parametrises each Gaussian; the tiny difference comes from the variance floor, which is defined relative to the largest variance). **Rule of thumb: always standardise before any distance-based method**, and always do it inside a `Pipeline` so test/validation data never leaks into the scaler.

## 7. Choosing $k$ with Cross-Validation

For each odd $k$ from 1 to 51 we estimate accuracy with 5×3 repeated stratified CV on the **training set**. The spread across folds tells us how much of an apparent difference between two $k$ values is just noise.

We then use two selection rules:

* **Best-mean rule:** the $k$ with the highest mean CV accuracy.
* **One-standard-error rule:** the *largest* $k$ (smoothest, simplest model) whose mean is within one standard error of the best. This guards against picking a $k$ that wins by luck.

In [ ]:
# ── Cross-Validated Sweep over k ──
ks = np.arange(1, 52, 2)
means, sems = [], []
for k in ks:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=int(k)))
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    means.append(s.mean()); sems.append(s.std(ddof=1) / np.sqrt(len(s)))
means, sems = np.array(means), np.array(sems)

best_i = means.argmax()
k_best = int(ks[best_i])
within = np.where(means >= means[best_i] - sems[best_i])[0]
k_1se = int(ks[within.max()])
nb_cv_mean, _ = cv_acc(make_pipeline(StandardScaler(), GaussianNB()))

# Training accuracy for the bias/variance picture (fit on full train set, scored on itself)
train_acc = [make_pipeline(StandardScaler(), KNeighborsClassifier(int(k))).fit(X_train, y_train).score(X_train, y_train) for k in ks]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ks, train_acc, "o--", color="gray", alpha=0.7, label="Train accuracy (optimistic)")
ax.plot(ks, means, "o-", color=C_KNN, label="k-NN CV accuracy")
ax.fill_between(ks, means - sems, means + sems, color=C_KNN, alpha=0.2, label="± 1 s.e.")
ax.axhline(nb_cv_mean, color=C_NB, ls=":", lw=2, label=f"Gaussian NB CV ({nb_cv_mean:.3f})")
ax.axvline(k_best, color="k", ls="--", lw=1, label=f"best mean: k={k_best}")
ax.axvline(k_1se, color="purple", ls="--", lw=1, label=f"1-s.e. rule: k={k_1se}")
ax.set_xlabel("k (number of neighbours)"); ax.set_ylabel("Accuracy"); ax.set_ylim(0.9, 1.01)
ax.set_title("Choosing k by cross-validation (training set only)"); ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

print(f"Best mean CV accuracy: k={k_best} ({means[best_i]:.4f}) | 1-s.e. rule picks k={k_1se} ({means[ks == k_1se][0]:.4f})")
K = k_1se
print(f"CV accuracy range over all k: {means.min():.3f} – {means.max():.3f} | k=1: {means[0]:.3f}, k=51: {means[-1]:.3f}")
print(f"→ We use k = {K} for the rest of the notebook.")

**Reading the curve:** at $k=1$ the training accuracy is exactly 1.0 (every point is its own nearest neighbour) while CV accuracy is lower: pure memorisation. As $k$ grows the two curves approach each other. Notice also how flat the CV curve is across many $k$ values: differences of a few tenths of a percent are *within the noise band*, which is exactly why the 1-s.e. rule prefers the simpler model.

## 8. Irrelevant Features: The Curse of Dimensionality

In high dimensions, all points become roughly equally far from each other, so "nearest" loses meaning, and irrelevant features add pure noise to every distance. We test this directly: append $m$ **pure-noise** columns (standard normal, unrelated to the label) to the 30 real features and track CV accuracy for both models.

In [ ]:
# ── Adding Irrelevant Noise Features ──
rng = np.random.default_rng(SEED)
noise_counts = [0, 10, 30, 100, 300, 1000]
res = {"k-NN": [], "Gaussian NB": []}
for m in noise_counts:
    Xn = np.hstack([X_train, rng.standard_normal((len(X_train), m))]) if m else X_train
    res["k-NN"].append(cv_acc(make_pipeline(StandardScaler(), KNeighborsClassifier(K)), Xn)[0])
    res["Gaussian NB"].append(cv_acc(make_pipeline(StandardScaler(), GaussianNB()), Xn)[0])

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(noise_counts, res["k-NN"], "o-", color=C_KNN, label=f"k-NN (k={K})")
ax.plot(noise_counts, res["Gaussian NB"], "s-", color=C_NB, label="Gaussian NB")
ax.set_xscale("symlog", linthresh=10); ax.set_xticks(noise_counts); ax.set_xticklabels(noise_counts)
ax.set_xlabel("Number of added pure-noise features (30 real features always present)")
ax.set_ylabel("CV accuracy"); ax.set_title("Robustness to irrelevant features"); ax.legend()
plt.tight_layout(); plt.show()

print(f"{'noise cols':>10} | {'k-NN':>7} | {'NB':>7}")
for m, a, b in zip(noise_counts, res['k-NN'], res['Gaussian NB']):
    print(f"{m:>10} | {a:>7.4f} | {b:>7.4f}")

**Why the difference?** k-NN weights every feature equally in the distance, so a thousand noise columns drown the 30 informative ones. Naïve Bayes handles each feature separately: a noise feature has (nearly) the same class-conditional Gaussian for both classes, so its log-likelihood ratio is close to zero and contributes little. (With finite data each noise feature still adds a small random push, so it is not guaranteed to be immune, but in this experiment its accuracy barely moves.) (In practice you would use **feature selection** or distance learning with k-NN.)

## 9. Correlated Features: When "Naïve" Hurts

The opposite problem: *redundant* features. If the same evidence appears several times, Naïve Bayes **counts it several times**, since it multiplies the likelihoods as if each copy were new, independent information. That typically leaves the predicted *class* mostly unchanged but pushes the predicted *probabilities* towards 0 and 1 (overconfidence).

k-NN is unaffected by exact duplicates after scaling: every distance is multiplied by the same constant $\sqrt{r}$, so the neighbour ranking is unchanged. We replicate the whole feature matrix $r$ times and measure both **accuracy** and **log-loss** (which punishes confident mistakes).

In [ ]:
# ── Duplicating Features: Accuracy vs. Probability Quality ──
copies = [1, 2, 4, 8, 16]
out = {"k-NN": {"acc": [], "ll": []}, "Gaussian NB": {"acc": [], "ll": []}}
cv5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
for r in copies:
    Xr = np.tile(X_train, (1, r))
    for name, model in [("k-NN", make_pipeline(StandardScaler(), KNeighborsClassifier(K))),
                        ("Gaussian NB", make_pipeline(StandardScaler(), GaussianNB()))]:
        P = cross_val_predict(model, Xr, y_train, cv=cv5, method="predict_proba")
        P = np.clip(P, 1e-6, 1 - 1e-6)
        out[name]["acc"].append(accuracy_score(y_train, (P[:, 1] > 0.5).astype(int)))
        out[name]["ll"].append(log_loss(y_train, P))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for name, col in [("k-NN", C_KNN), ("Gaussian NB", C_NB)]:
    axes[0].plot(copies, out[name]["acc"], "o-", color=col, label=name)
    axes[1].plot(copies, out[name]["ll"], "o-", color=col, label=name)
for ax, t in zip(axes, ["Accuracy (out-of-fold)", "Log-loss (lower = better probabilities)"]):
    ax.set_xscale("log", base=2); ax.set_xticks(copies); ax.set_xticklabels(copies)
    ax.set_xlabel("Copies of the feature set (r)"); ax.set_title(t); ax.legend()
plt.suptitle("Redundant features: accuracy stays similar, NB probabilities degrade", y=1.02)
plt.tight_layout(); plt.show()
for name in out:
    print(f"{name:<12} accuracy r=1 → r=16: {out[name]['acc'][0]:.3f} → {out[name]['acc'][-1]:.3f} | "
          f"log-loss: {out[name]['ll'][0]:.3f} → {out[name]['ll'][-1]:.3f}")

**Takeaway:** Naïve Bayes can still be a good *ranker/classifier* under correlated features, but its probabilities are not trustworthy when features are redundant. If you need probabilities (risk scores, decision thresholds), check calibration (Section 11) or use a discriminative model such as logistic regression.

## 10. Beyond Accuracy: Confusion Matrix, ROC/AUC, Threshold

Accuracy hides *which* errors a model makes. In a screening-style problem a **false negative** (a missed malignant case) is usually far more costly than a **false positive**. We now fit the final models on the training set (with the CV-chosen $k$) and evaluate **once** on the held-out test set.

* **Sensitivity (recall)** $= \frac{TP}{TP+FN}$: fraction of malignant cases caught.
* **Specificity** $= \frac{TN}{TN+FP}$: fraction of benign cases correctly cleared.
* **ROC curve:** sensitivity vs. $1-$specificity across *all* thresholds; **AUC** is the probability that a random malignant case is scored above a random benign one.

In [ ]:
# ── Final Models, Test-Set Evaluation ──
final = {
    f"k-NN (k={K})": make_pipeline(StandardScaler(), KNeighborsClassifier(K)).fit(X_train, y_train),
    "Gaussian NB":   make_pipeline(StandardScaler(), GaussianNB()).fit(X_train, y_train),
}
colors = {f"k-NN (k={K})": C_KNN, "Gaussian NB": C_NB}
proba = {n: m.predict_proba(X_test)[:, 1] for n, m in final.items()}
pred  = {n: m.predict(X_test) for n, m in final.items()}

for n in final:
    print(f"=== {n} ===")
    print(classification_report(y_test, pred[n], target_names=["benign", "malignant"], digits=3))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, n in zip(axes[:2], final):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred[n]), display_labels=["benign", "malignant"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{n}\nacc = {accuracy_score(y_test, pred[n]):.3f}")
for n in final:
    fpr, tpr, _ = roc_curve(y_test, proba[n])
    axes[2].plot(fpr, tpr, color=colors[n], lw=2, label=f"{n} (AUC = {roc_auc_score(y_test, proba[n]):.3f})")
axes[2].plot([0, 1], [0, 1], "k--", lw=1, label="chance")
axes[2].set_xlabel("False positive rate (1 − specificity)"); axes[2].set_ylabel("True positive rate (sensitivity)")
axes[2].set_title("ROC curves (test set)"); axes[2].legend(loc="lower right")
plt.tight_layout(); plt.show()

### Choosing an operating threshold

The default cut-off of 0.5 is arbitrary. If missing a malignant case is costly, we can lower the threshold on the predicted probability, trading false positives for fewer false negatives. **The threshold must be chosen without looking at the test set.** We therefore pick it from *out-of-fold* training predictions (target: sensitivity ≥ 95%) and then apply it once to the test set.

In [ ]:
# ── Threshold Chosen by CV, Applied Once to Test ──
TARGET_SENS = 0.95
print(f"{'Model':<14}{'thr':>9}{'  FN (default → tuned)':>24}{'  FP (default → tuned)':>24}")
for n, model_name in zip(final, ["knn", "nb"]):
    base = (make_pipeline(StandardScaler(), KNeighborsClassifier(K)) if model_name == "knn"
            else make_pipeline(StandardScaler(), GaussianNB()))
    p_oof = cross_val_predict(base, X_train, y_train, cv=cv5, method="predict_proba")[:, 1]
    fpr, tpr, thr = roc_curve(y_train, p_oof)
    ok = np.where(tpr >= TARGET_SENS)[0]
    t = float(thr[ok[0]])                       # highest threshold that still reaches the target sensitivity
    t = min(t, 0.5)                             # never raise the threshold above the default
    tuned = (proba[n] >= t).astype(int)
    tn0, fp0, fn0, tp0 = confusion_matrix(y_test, pred[n]).ravel()
    tn1, fp1, fn1, tp1 = confusion_matrix(y_test, tuned).ravel()
    print(f"{n:<14}{t:>9.4f}{f'{fn0} → {fn1}':>24}{f'{fp0} → {fp1}':>24}")
print("\nLowering the threshold catches more malignant cases (fewer FN) at the price of more false alarms (FP).")
print("Note: with k=3, k-NN can only output 0, 1/3, 2/3 or 1, so its threshold choices are very coarse; larger k gives finer control.")

## 11. Probability Calibration

A model is **calibrated** if, among samples it assigns probability $p$, a fraction $p$ are truly positive. This matters whenever the probability itself is used (risk scores, expected-cost decisions).

* **k-NN** probabilities are vote fractions $\in \{0, 1/k, \dots, 1\}$, which are coarse (with small $k$ many predictions are exactly 0 or 1).
* **Naïve Bayes** probabilities tend to be pushed toward 0 and 1 because correlated evidence is multiplied together (Section 9).

We compare using a **reliability diagram** and the **Brier score** (mean squared error of the probability; lower is better).

In [ ]:
# ── Reliability Diagram + Brier Score + Confidence Histogram ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].plot([0, 1], [0, 1], "k--", label="perfectly calibrated")
for n in final:
    frac, mean_p = calibration_curve(y_test, proba[n], n_bins=6, strategy="quantile")
    axes[0].plot(mean_p, frac, "o-", color=colors[n], label=f"{n} (Brier = {brier_score_loss(y_test, proba[n]):.3f})")
    axes[1].hist(proba[n], bins=20, range=(0, 1), alpha=0.55, color=colors[n], label=n)
axes[0].set_xlabel("Mean predicted P(malignant)"); axes[0].set_ylabel("Observed fraction malignant")
axes[0].set_title("Reliability diagram (test set)"); axes[0].legend(fontsize=9)
axes[1].set_xlabel("Predicted P(malignant)"); axes[1].set_ylabel("Count"); axes[1].set_yscale("log")
axes[1].set_title("How confident is each model? (log-scale counts)"); axes[1].legend()
plt.tight_layout(); plt.show()

for n in final:
    p = proba[n]; extreme = ((p < 0.02) | (p > 0.98)).mean()
    print(f"{n:<14}: {extreme*100:5.1f}% of test predictions have P<0.02 or P>0.98")

With only ~143 test points the reliability curve is noisy, so read it as a trend, not a precise measurement (a good exercise: repeat it with cross-validated out-of-fold predictions on all 569 samples). Compare the histograms: k-NN probabilities live on a coarse grid of vote fractions, while Naïve Bayes concentrates its mass at the extremes.

## 12. Learning Curves

How much data does each model need? A **learning curve** plots cross-validated accuracy against training-set size. A textbook expectation (Ng & Jordan, 2001) is that a simple generative model such as Naïve Bayes can reach its (higher-bias) asymptote with *less* data, while a more flexible model needs more data but may end up better. Whether your data follow that pattern is an empirical question, so we measure it.

In [ ]:
# ── Learning Curves ──
sizes = np.linspace(0.05, 1.0, 10)
cv_lc = StratifiedKFold(5, shuffle=True, random_state=SEED)
fig, ax = plt.subplots(figsize=(9, 5))
for n, mdl, col in [(f"k-NN (k={K})", make_pipeline(StandardScaler(), KNeighborsClassifier(K)), C_KNN),
                    ("Gaussian NB", make_pipeline(StandardScaler(), GaussianNB()), C_NB)]:
    ts, tr, va = learning_curve(mdl, X_train, y_train, train_sizes=sizes, cv=cv_lc,
                                scoring="accuracy", shuffle=True, random_state=SEED)
    ax.plot(ts, va.mean(1), "o-", color=col, label=f"{n}: validation")
    ax.fill_between(ts, va.mean(1) - va.std(1), va.mean(1) + va.std(1), color=col, alpha=0.15)
    ax.plot(ts, tr.mean(1), "--", color=col, alpha=0.6, label=f"{n}: train")
ax.set_xlabel("Number of training samples"); ax.set_ylabel("Accuracy"); ax.set_ylim(0.8, 1.02)
ax.set_title("Learning curves (shaded = ±1 std across folds)"); ax.legend(fontsize=9, loc="lower right")
plt.tight_layout(); plt.show()

**Questions to ask of this plot:** Do the train and validation curves converge (low variance) or stay apart (high variance)? Does either model's validation curve still rise at the right edge (would more data help)? Which model is better with only ~25 samples, and does the order flip later?

## 13. Summary

| Topic | What we found |
|---|---|
| **Boundaries** | NB is limited to smooth, at-most-quadratic boundaries (fine for circles, poor for moons); k-NN follows the data. |
| **From scratch** | Both models are ~20 lines of NumPy and match scikit-learn exactly. |
| **Scaling** | Essential for k-NN, (almost) irrelevant for NB. Always standardise inside a `Pipeline`. |
| **Choosing $k$** | Use repeated CV on the training set; prefer the 1-s.e. rule; never tune on the test set. |
| **Irrelevant features** | k-NN suffers from the curse of dimensionality; NB is far more robust. |
| **Correlated features** | NB double-counts evidence: classification survives, probabilities do not. |
| **Evaluation** | Use confusion matrix, ROC/AUC, calibration and a CV-chosen threshold, not accuracy alone. |

**Rules of thumb:** many noisy / high-dimensional features or tiny data → try NB (or a regularised linear model). Few, well-scaled, informative features and a non-Gaussian shape → k-NN. If you need good probabilities or interpretability, compare with logistic regression (see `naive_bayes_vs_logreg_tutorial.ipynb` in this folder).

## 14. Questions & Bonus Exercises

### 💬 Discussion Questions

**Q1.** In Section 3, Gaussian NB handles the concentric circles well but struggles on the two moons. Explain why, using the fact that each class gets its own mean *and* variance per feature. What would happen on circles if both classes had the same variance? Test your answer by changing `factor` in `make_circles`.

**Q2.** At $k=1$ the training accuracy is always 1. Why? What would training accuracy be for $k$ = the whole training set size?

**Q3.** Section 6 shows NB is unaffected by standardisation. Prove it: show that replacing $x_j$ by $a x_j + b$ changes $\mu_{c,j}, \sigma^2_{c,j}$ in a way that leaves the posterior unchanged.

**Q4.** In our run both selection rules happened to pick the same $k$. In general, why can the one-standard-error rule select a *larger* $k$ than the best-mean rule, and when would you prefer the best-mean $k$? Re-run with a different `SEED` and see whether they still agree.

**Q5.** In Section 8, explain in terms of distances why 1,000 noise features hurt k-NN. Roughly what happens to the ratio between the nearest and farthest neighbour distance as $p \to \infty$?

**Q6.** In a screening setting, would you rather report accuracy, sensitivity, or AUC? What clinical/economic information would you need to pick an operating threshold?

**Q7.** Section 9 shows NB probabilities get worse with duplicated features, yet accuracy barely changes. Why does the argmax survive while the probabilities do not?

---

### 🚀 Bonus Exercises

**Bonus 1 — Distance weighting.** Extend `KNNScratch` with `weights="distance"` (vote weight $1/d$) and compare CV accuracy to uniform voting across several $k$.

**Bonus 2 — Other metrics.** Swap Euclidean for Manhattan ($L_1$) or cosine distance. Which holds up better when you add noise features (Section 8)?

**Bonus 3 — Calibrate NB.** Wrap Naïve Bayes in `CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5)` and redo Section 11. Does the Brier score improve? What happens to AUC?

**Bonus 4 — Feature selection for k-NN.** Use `SelectKBest(f_classif, k=10)` *inside* the pipeline. Repeat Section 8: how much of k-NN's robustness to noise features can you recover?

**Bonus 5 — Decorrelate for NB.** Apply `PCA(whiten=True)` before Naïve Bayes. Does it help accuracy or log-loss on the original (correlated) features? Why might PCA make the independence assumption *more* plausible?

**Bonus 6 — Speed.** Time `fit` and `predict` for both models on $n \in \{10^3, 10^4, 10^5\}$ synthetic points. Plot the results and also try `KNeighborsClassifier(algorithm="kd_tree")`. When do trees help, and when do they stop helping?

**Bonus 7 — Nested CV.** The $k$ selection in Section 7 and its performance estimate use the same CV folds, which is slightly optimistic. Implement *nested* cross-validation with `GridSearchCV` inside `cross_val_score` and compare.

### 📚 References
* Cover & Hart (1967), *Nearest neighbor pattern classification*.
* Ng & Jordan (2001), *On discriminative vs. generative classifiers: a comparison of logistic regression and naive Bayes*.
* Street, Wolberg & Mangasarian (1993), *Nuclear feature extraction for breast tumor diagnosis* (the Wisconsin dataset).
* Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*, Ch. 2, 6 and 13.